[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/12_montecarlo/notebooks/01_fundamentos.ipynb)

# Notebook 1: Fundamentos de Monte Carlo

En las notas leíste la historia y los fundamentos matemáticos. Ahora toca **vivirlos**.

Este notebook tiene cuatro secciones:

1. **Estimación de π** — el truco clásico para entender la mecánica
2. **Integración general** — de 1D a 20D y por qué MC no se rompe
3. **LLN + CLT en acción** — por qué funciona, en código
4. **Primer vistazo a reducción de varianza** — mismo presupuesto, mejor precisión

**Cómo usar este notebook:**  
Lee cada celda de texto, luego ejecuta el código. Cuando veas `# <-- CHANGE THIS`, ese es tu momento para experimentar. Los ejercicios están marcados con 🔧.

In [ ]:
# Descomenta si estás en Colab
# !pip install numpy matplotlib scipy --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

rng = np.random.default_rng(42)
print("Imports listos ✓")

---
## Sección 1: Estimación de π

### La idea geométrica

Considera un cuadrado de lado 2 centrado en el origen $[-1,1]^2$ y el círculo unitario $x^2 + y^2 \leq 1$ inscrito en él.

- Área del cuadrado: $4$
- Área del círculo: $\pi$
- Proporción: $\pi / 4$

Si lanzamos puntos **uniformemente al azar** dentro del cuadrado, la probabilidad de que un punto caiga dentro del círculo es exactamente $\pi/4$.

$$P(\text{punto dentro del círculo}) = \frac{\pi}{4}$$

Entonces:

$$\hat{\pi} = 4 \times \frac{\text{puntos dentro del círculo}}{\text{total de puntos}}$$

Esto es Monte Carlo en su forma más pura: **muestrea y promedia**. El estimador que usamos es:

$$\hat{\mu}_n = \frac{1}{n} \sum_{i=1}^n f(X_i), \qquad f(x,y) = \mathbf{1}[x^2 + y^2 \leq 1], \quad (X,Y) \sim \text{Uniform}([-1,1]^2)$$

y $\pi = 4 \cdot \mathbb{E}[f(X,Y)]$.

In [ ]:
def estimate_pi(n, seed=None):
    """Estimate pi using n random points in [-1,1]^2."""
    rng_local = np.random.default_rng(seed)
    x = rng_local.uniform(-1, 1, n)
    y = rng_local.uniform(-1, 1, n)
    inside = x**2 + y**2 <= 1.0
    return 4 * inside.mean(), x, y, inside

# Visualize for one value of n
n_vis = 2000  # <-- CHANGE THIS
pi_est, x, y, inside = estimate_pi(n_vis, seed=42)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x[~inside], y[~inside], s=2, color="#7F8C8D", alpha=0.4, label="Fuera")
ax.scatter(x[inside], y[inside], s=2, color="#2E86AB", alpha=0.6, label="Dentro")
theta = np.linspace(0, 2*np.pi, 400)
ax.plot(np.cos(theta), np.sin(theta), "r-", lw=1.5)
ax.set_aspect("equal")
ax.set_title(f"$n={n_vis:,}$ | $\\hat{{\\pi}}={pi_est:.4f}$ | error = {abs(pi_est-np.pi):.4f}")
ax.legend(markerscale=5)
plt.tight_layout()
plt.show()

In [ ]:
# How does the estimate improve with n?
ns = [100, 1_000, 10_000, 100_000]  # <-- CHANGE THIS

print(f"{'n':>10} | {'π̂':>10} | {'error':>10} | {'error/prev':>12}")
print("-" * 50)
prev_error = None
for n in ns:
    pi_hat, *_ = estimate_pi(n, seed=n)  # different seed per n for independence
    error = abs(pi_hat - np.pi)
    ratio = f"{error/prev_error:.2f}x" if prev_error else "—"
    print(f"{n:>10,} | {pi_hat:>10.5f} | {error:>10.5f} | {ratio:>12}")
    prev_error = error

### 🔧 Ejercicio: ¿Cuántos samples necesitas para 3 decimales correctos?

Queremos que el error sea $|\hat{\pi} - \pi| < 0.001$ con **95% de confianza**.

Del Teorema Central del Límite sabemos que para $n$ grande:

$$\hat{\pi}_n \approx \mathcal{N}\left(\pi,\ \frac{\sigma^2}{n}\right)$$

donde $\sigma^2 = \text{Var}[4 \cdot \mathbf{1}[X^2+Y^2 \leq 1]]$.

**Paso 1:** Estima $\sigma$ empíricamente (puedes usar muchos samples y calcular la std de los $f(X_i) = 4 \cdot \mathbf{1}[\cdot]$).

**Paso 2:** Para que el IC al 95% tenga radio $\varepsilon$, necesitas:
$$n \geq \left(\frac{z_{0.025} \cdot \sigma}{\varepsilon}\right)^2$$

**Paso 3:** Verifica corriendo `estimate_pi` con ese $n$ muchas veces y cuenta qué fracción tiene error $< 0.001$.

In [ ]:
# Step 1: Estimate sigma
n_large = 1_000_000
x_big = rng.uniform(-1, 1, n_large)
y_big = rng.uniform(-1, 1, n_large)
f_vals = 4 * (x_big**2 + y_big**2 <= 1).astype(float)
sigma_hat = f_vals.std()
print(f"σ̂ = {sigma_hat:.4f}")

# Step 2: Required n
epsilon = 0.001  # <-- CHANGE THIS to try different precisions
z = 1.96
n_required = int(np.ceil((z * sigma_hat / epsilon) ** 2))
print(f"n necesario para error < {epsilon} con 95% confianza: {n_required:,}")

# Step 3: Empirical verification
n_trials = 200
successes = sum(
    abs(estimate_pi(n_required, seed=i)[0] - np.pi) < epsilon
    for i in range(n_trials)
)
print(f"Fracción de éxito empírica: {successes}/{n_trials} = {successes/n_trials:.2%}")
print(f"(Esperada: ~95%)")

---
## Sección 2: Integración General

### Del círculo a cualquier integral

La estimación de π es un caso especial de la idea general: **Monte Carlo puede estimar cualquier integral**.

Si queremos calcular:
$$I = \int_a^b f(x)\, dx$$

podemos escribirlo como una esperanza. Si $X \sim \text{Uniform}(a, b)$, entonces:

$$I = (b-a) \cdot \mathbb{E}[f(X)]$$

El estimador MC es:

$$\hat{I}_n = \frac{b-a}{n} \sum_{i=1}^n f(X_i), \qquad X_i \sim \text{Uniform}(a,b)$$

En $d$ dimensiones sobre el hipercubo $[0,1]^d$:

$$I_d = \int_{[0,1]^d} f(\mathbf{x})\, d\mathbf{x} \approx \frac{1}{n} \sum_{i=1}^n f(\mathbf{X}_i), \qquad \mathbf{X}_i \sim \text{Uniform}([0,1]^d)$$

**Nótese**: el estimador es exactamente el mismo en $d=1$ y en $d=100$. Solo cambia de dónde muestreamos $\mathbf{X}_i$.

In [ ]:
# Example: integrate f(x) = exp(-x^2) over [0, 1]
# True value: erf(1) * sqrt(pi)/2 ≈ 0.7468
from scipy.special import erf

true_val_1d = np.sqrt(np.pi) / 2 * erf(1)
print(f"Valor verdadero ∫₀¹ exp(-x²)dx = {true_val_1d:.6f}")

def mc_integrate_1d(f, n, seed=None):
    """MC estimate of integral of f over [0,1]."""
    rng_local = np.random.default_rng(seed)
    x = rng_local.uniform(0, 1, n)
    return f(x).mean()

f = lambda x: np.exp(-x**2)

print("\nEstimaciones MC:")
for n in [100, 1_000, 10_000, 100_000]:
    est = mc_integrate_1d(f, n, seed=n)
    print(f"  n={n:>8,}: {est:.6f}  (error={abs(est-true_val_1d):.6f})")

### La potencia en alta dimensión

Considera integrar $f(\mathbf{x}) = x_1^2 + x_2^2 + \cdots + x_d^2$ sobre el hipercubo $[0,1]^d$.

El valor verdadero es fácil de calcular por linealidad:

$$\int_{[0,1]^d} \sum_{j=1}^d x_j^2\, d\mathbf{x} = \sum_{j=1}^d \int_0^1 x_j^2\, dx_j = d \cdot \frac{1}{3} = \frac{d}{3}$$

Ahora veamos qué pasaría con un **método de cuadratura** (rejilla uniforme):
- En $d$ dimensiones con $m$ puntos por dimensión, necesitamos $m^d$ puntos totales
- Para error $\varepsilon$ con un método de cuadratura simple: $m \sim \varepsilon^{-1/2}$, total $\sim \varepsilon^{-d/2}$
- Para $\varepsilon = 0.01$ y $d = 20$: necesitas $100^{20} = 10^{40}$ puntos. **Imposible.**

Con Monte Carlo, el mismo $n = 10{,}000$ puntos funciona en cualquier dimensión.

In [ ]:
def mc_integrate_nd(d, n, seed=None):
    """MC estimate of integral of sum(x_j^2) over [0,1]^d."""
    rng_local = np.random.default_rng(seed)
    X = rng_local.uniform(0, 1, (n, d))
    f_vals = (X**2).sum(axis=1)   # sum of squares per point
    return f_vals.mean()

n = 10_000  # <-- CHANGE THIS
dims = [1, 2, 5, 10, 20]  # <-- CHANGE THIS

print(f"{'d':>5} | {'Verdadero':>10} | {'MC (n={n:,})':>14} | {'Error':>10}")
print("-" * 50)
for d in dims:
    true = d / 3
    est = mc_integrate_nd(d, n, seed=d)
    print(f"{d:>5} | {true:>10.4f} | {est:>14.4f} | {abs(est-true):>10.4f}")

In [ ]:
# Visual comparison: error of MC vs quadrature as a function of dimension
dims_range = np.arange(1, 16)
n_mc = 10_000
epsilon = 0.01

# MC: constant cost
n_mc_line = np.full_like(dims_range, n_mc, dtype=float)

# Quadrature: exponential in d (m=10 pts/dim → m^d total)
m_per_dim = 10  # 10 points per dimension
n_quad_line = m_per_dim ** dims_range.astype(float)

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(dims_range, n_quad_line, "o-", color="#E94F37", lw=2, ms=6,
            label=f"Cuadratura ({m_per_dim} pts/dim): $n = {m_per_dim}^d$")
ax.semilogy(dims_range, n_mc_line, "s--", color="#2E86AB", lw=2, ms=6,
            label=f"Monte Carlo: $n = {n_mc:,}$ (constante)")

# Crossover
cross = dims_range[n_mc_line >= n_quad_line]
if len(cross) > 0:
    ax.axvline(cross[0], color="gray", ls=":", lw=1.5,
               label=f"Cruce en $d \\approx {cross[0]}$")

ax.set_xlabel("Dimensión $d$")
ax.set_ylabel("Puntos necesarios (escala log)")
ax.set_title("Costo en función de la dimensión\nMC no sufre la maldición de la dimensionalidad")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 🔧 Ejercicio: Hipercubo 10D

Estima $\int_{[0,1]^{10}} (x_1^2 + \cdots + x_{10}^2)\, d\mathbf{x}$ con Monte Carlo.

1. **Verifica analíticamente**: ¿cuál es el valor exacto? (usa la linealidad que derivamos arriba)
2. **Experimenta**: ¿Cuántos samples necesitas para que el error sea menor a 0.01 consistentemente?
3. **Cuadratura**: si usaras una rejilla con 10 puntos por dimensión, ¿cuántos puntos en total? ¿Es manejable?
4. **Reto**: intenta estimar $\int_{[0,1]^{50}} \sum_j x_j^2\, d\mathbf{x}$ y verifica contra el valor exacto.

In [ ]:
d = 10  # <-- CHANGE THIS
n = 50_000  # <-- CHANGE THIS

true_val = d / 3
mc_est = mc_integrate_nd(d, n, seed=42)
print(f"Valor verdadero: {true_val:.6f}")
print(f"Estimado MC (n={n:,}): {mc_est:.6f}")
print(f"Error: {abs(mc_est - true_val):.6f}")
print(f"\nPuntos cuadratura (10 pts/dim): {10**d:.2e}")

---
## Sección 3: LLN y CLT en Acción

Hasta ahora hemos *usado* Monte Carlo. Ahora vamos a *entender* por qué funciona.

### La Ley de los Grandes Números

La LLN dice que el promedio muestral converge al valor esperado:

$$\hat{\mu}_n = \frac{1}{n}\sum_{i=1}^n f(X_i) \xrightarrow{\text{c.s.}} \mathbb{E}[f(X)] = \mu$$

**Supuesto**: $\mathbb{E}[|f(X)|] < \infty$ (el valor esperado existe).

Veámoslo visualmente: el promedio acumulado empieza caótico y se va estabilizando.

In [ ]:
def lln_demo(f, true_val, n_max=50_000, n_runs=5, label=""):
    """Plot running average of f(X) for n_runs independent sequences."""
    fig, ax = plt.subplots(figsize=(10, 4))
    ns = np.arange(1, n_max + 1)
    colors = ["#2E86AB", "#E94F37", "#27AE60", "#F39C12", "#8E44AD"]

    for i in range(n_runs):
        rng_local = np.random.default_rng(i)
        x = rng_local.uniform(0, 1, n_max)
        running_avg = np.cumsum(f(x)) / ns
        ax.semilogx(ns, running_avg, lw=0.8, color=colors[i], alpha=0.7,
                    label=f"Corrida {i+1}")

    ax.axhline(true_val, color="black", lw=1.8, ls="--", label=f"Valor verdadero = {true_val:.4f}")
    ax.set_xlabel("Número de muestras $n$ (log)")
    ax.set_ylabel("Promedio acumulado")
    ax.set_title(f"Ley de los Grandes Números: convergencia de {label}")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

# Estimating pi: f(u) = 4*sqrt(1-u^2), E[f] = pi
lln_demo(
    f=lambda u: 4 * np.sqrt(1 - u**2),
    true_val=np.pi,
    n_runs=4,
    label="$\\hat{\\pi}_n$ (estimando π)"
)

### El Teorema Central del Límite

La LLN nos dice que $\hat{\mu}_n \to \mu$. El CLT nos dice cómo está distribuido $\hat{\mu}_n$ alrededor de $\mu$:

$$\sqrt{n}(\hat{\mu}_n - \mu) \xrightarrow{d} \mathcal{N}(0, \sigma^2)$$

**Supuesto**: $\text{Var}[f(X)] = \sigma^2 < \infty$.

Para $n$ grande: $\hat{\mu}_n \approx \mathcal{N}(\mu, \sigma^2/n)$.

Podemos verificar esto empíricamente: repetimos el experimento muchas veces y miramos la distribución de los estimados.

In [ ]:
def clt_demo(f, true_val, ns_to_show, n_experiments=2000):
    """Show distribution of MC estimates for different n values."""
    # Estimate sigma
    rng_local = np.random.default_rng(0)
    sigma = f(rng_local.uniform(0, 1, 500_000)).std()

    fig, axes = plt.subplots(1, len(ns_to_show), figsize=(5*len(ns_to_show), 4), sharey=False)
    colors = ["#2E86AB", "#F39C12", "#27AE60"]

    for ax, n, color in zip(axes, ns_to_show, colors):
        estimates = np.array([
            f(np.random.default_rng(i).uniform(0, 1, n)).mean()
            for i in range(n_experiments)
        ])
        se = sigma / np.sqrt(n)
        ax.hist(estimates, bins=50, density=True, color=color, alpha=0.6,
                edgecolor="white", lw=0.3)

        # Theoretical normal
        x_range = np.linspace(estimates.min(), estimates.max(), 300)
        ax.plot(x_range, stats.norm.pdf(x_range, true_val, se),
                "k-", lw=2, label=f"$\\mathcal{{N}}(\\mu, \\sigma^2/{n})$")
        ax.axvline(true_val, color="red", ls="--", lw=1.5)
        ax.set_title(f"$n={n:,}$\n$\\sigma_{{\\hat{{\\mu}}}} = {se:.4f}$")
        ax.set_xlabel("$\\hat{\\mu}_n$")
        ax.legend(fontsize=8)

    fig.suptitle(f"CLT: distribución de {n_experiments} estimados independientes", y=1.02)
    plt.tight_layout()
    plt.show()

clt_demo(
    f=lambda u: 4 * np.sqrt(1 - u**2),
    true_val=np.pi,
    ns_to_show=[10, 100, 1000]  # <-- CHANGE THIS
)

### Construyendo el Intervalo de Confianza paso a paso

Del CLT sabemos que:
$$\hat{\mu}_n \approx \mathcal{N}\left(\mu,\ \frac{\sigma^2}{n}\right)$$

Un IC al 95% para $\mu$ es:
$$\left[\hat{\mu}_n - 1.96 \cdot \frac{\hat{\sigma}}{\sqrt{n}},\ \ \hat{\mu}_n + 1.96 \cdot \frac{\hat{\sigma}}{\sqrt{n}}\right]$$

donde $\hat{\sigma}$ es la std muestral de los $f(X_i)$. En código:

In [ ]:
def mc_estimate_with_ci(f, n, alpha=0.05, seed=42):
    """MC estimate with confidence interval."""
    rng_local = np.random.default_rng(seed)
    x = rng_local.uniform(0, 1, n)
    fvals = f(x)

    mu_hat = fvals.mean()
    sigma_hat = fvals.std(ddof=1)
    z = stats.norm.ppf(1 - alpha/2)  # 1.96 for alpha=0.05
    margin = z * sigma_hat / np.sqrt(n)

    return mu_hat, mu_hat - margin, mu_hat + margin

# Apply to pi estimation
f_pi = lambda u: 4 * np.sqrt(1 - u**2)

print(f"Estimando π con intervalos de confianza al 95%:")
print(f"{'n':>10} | {'Estimado':>10} | {'IC inferior':>12} | {'IC superior':>12} | {'Contiene π?':>12}")
print("-" * 65)
for n in [100, 1_000, 10_000, 100_000]:
    est, lo, hi = mc_estimate_with_ci(f_pi, n, seed=n)
    contains = "✓" if lo <= np.pi <= hi else "✗"
    print(f"{n:>10,} | {est:>10.5f} | {lo:>12.5f} | {hi:>12.5f} | {contains:>12}")

### 🔧 Ejercicio desafiante: cuando el CLT falla

Considera $f(x) = 1/x$ cerca de $x = 0$, con $X \sim \text{Uniform}(\epsilon, 1)$.

1. ¿Qué pasa cuando $\epsilon \to 0$? ¿Sigue teniendo varianza finita $f$?
2. Experimenta con valores pequeños de $\epsilon$ y mira cómo se ve la distribución de los estimados
3. ¿El intervalo de confianza que calculamos es válido? ¿Por qué?

**Pista**: $\text{Var}[1/X]$ para $X \sim \text{Uniform}(\epsilon, 1)$ — ¿qué pasa cuando $\epsilon \to 0$?

In [ ]:
epsilon = 0.01  # <-- CHANGE THIS: try 0.1, 0.01, 0.001, 0.0001
n = 5_000
n_experiments = 500

# True value: integral of 1/x over [epsilon, 1] = -log(epsilon)
true_val_eps = -np.log(epsilon)
print(f"ε = {epsilon}, valor verdadero = {true_val_eps:.4f}")

estimates = []
for i in range(n_experiments):
    rng_local = np.random.default_rng(i)
    x = rng_local.uniform(epsilon, 1, n)
    # f(x) = 1/x, but we integrate over [epsilon,1] so need to scale
    fvals = (1 - epsilon) * (1 / x)  # (b-a) * f(X)
    estimates.append(fvals.mean())

estimates = np.array(estimates)
sigma_vals = []
for i in range(n_experiments):
    rng_local = np.random.default_rng(i)
    x = rng_local.uniform(epsilon, 1, n)
    sigma_vals.append(((1 - epsilon) * (1 / x)).std())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(estimates, bins=50, color="#2E86AB", alpha=0.7, edgecolor="white")
ax1.axvline(true_val_eps, color="red", ls="--", lw=2, label=f"Verdadero = {true_val_eps:.2f}")
ax1.set_title(f"Distribución de estimados\n$\\epsilon = {epsilon}$, $n = {n:,}$")
ax1.set_xlabel("Estimado")
ax1.legend()

ax2.plot(np.sort(sigma_vals), ".", ms=2, color="#E94F37")
ax2.set_title("$\\hat{\\sigma}$ muestral por experimento\n(si hay cola pesada, esto explota)")
ax2.set_xlabel("Experimento (ordenado)")
ax2.set_ylabel("$\\hat{\\sigma}$")

plt.tight_layout()
plt.show()
print(f"Skewness de los estimados: {stats.skew(estimates):.2f}")
print(f"Kurtosis de los estimados: {stats.kurtosis(estimates):.2f}")
print("\n(Para ε pequeño verás distribuciones muy sesgadas → el CLT no aplica bien)")

---
## Sección 3c: Colas Largas y Monte Carlo

> En el módulo 05 ([Colas Largas](../../../05_probabilidad/14_colas_largas.md)) vimos que las distribuciones fat-tailed rompen muchas suposiciones estadísticas. Aquí vemos exactamente qué le pasa a Monte Carlo.

### Recuerda: el supuesto que el CLT necesita

El CLT que usamos para construir ICs requiere:

$$\text{Var}[f(X)] = \sigma^2 < \infty$$

¿Qué pasa si este supuesto se viola? Depende de *cuánto* se viola.

### Los tres regímenes con Pareto($\alpha$)

Una distribución Pareto con exponente de cola $\alpha$ satisface $P(X > x) \sim x^{-\alpha}$.
El valor de $\alpha$ determina qué momentos existen:

| $\alpha$ | Media | Varianza | LLN | CLT | Para MC |
|:---:|:---:|:---:|:---:|:---:|---|
| $> 2$ | Finita | **Finita** | ✓ | ✓ | IC válido al 95% |
| $1 < \alpha \leq 2$ | Finita | **Infinita** | ✓ (lento) | ✗ | IC es **una ilusión** |
| $\leq 1$ (Cauchy) | Infinita | Infinita | ✗ | ✗ | Promedio no converge |

### ¿Qué se ve en la práctica?

- **$\alpha > 2$**: los estimados forman una campana limpia; el IC al 95% cubre ~95% de las veces
- **$\alpha = 1.5$**: los estimados tienen cola derecha pesada; la cobertura real del IC cae por debajo del 95%
- **$\alpha = 1.0$ (Cauchy)**: el promedio salta sin converger; cada nuevo punto extremo destruye el promedio anterior

### Señales de alarma en la práctica

Si ves estas señales en tu simulación Monte Carlo, sospecha colas pesadas:
- Los estimados cambian dramáticamente de una corrida a otra (con el mismo $n$)
- $\hat{\sigma}$ muestral es muy inestable — no se estabiliza al crecer $n$
- El IC al 95% no se estrecha como $1/\sqrt{n}$ cuando aumentas $n$

En los ejercicios de abajo puedes explorar todo esto cambiando `alpha`. Empieza con `alpha = 3.0` (funciona bien) y ve bajando hasta `alpha = 1.0` para ver la desintegración.

In [ ]:
# Comparación de coverage real del IC al 95% para distintas distribuciones
# Pregunta: ¿con qué frecuencia el IC que calculamos realmente contiene la media verdadera?

def pareto_standard(alpha, n, rng_local):
    """
    Pareto estándar con soporte [1, inf) y exponente de cola alpha.
    P(X > x) = x^{-alpha}, E[X] = alpha/(alpha-1) para alpha > 1.
    Var[X] = alpha / ((alpha-1)^2 * (alpha-2)) para alpha > 2.

    numpy.pareto(alpha) devuelve Z = X - 1 donde X es Pareto(alpha),
    así que X = numpy.pareto(alpha) + 1 es lo que queremos.
    """
    return rng_local.pareto(alpha, size=n) + 1

def coverage_experiment(alpha_pareto, n=500, n_trials=1000):
    if alpha_pareto <= 1:
        return np.nan, np.nan
    true_mean = alpha_pareto / (alpha_pareto - 1)
    covered = 0
    for seed in range(n_trials):
        rng_local = np.random.default_rng(seed + 1000)
        s = pareto_standard(alpha_pareto, n, rng_local)
        mu_hat = s.mean()
        sigma_hat = s.std(ddof=1)
        margin = 1.96 * sigma_hat / np.sqrt(n)
        if mu_hat - margin <= true_mean <= mu_hat + margin:
            covered += 1
    return covered / n_trials, true_mean

n = 500  # <-- CHANGE THIS (prueba 200, 500, 2000)
alphas = [5.0, 3.0, 2.5, 2.0, 1.8, 1.5]

print(f"Cobertura real del IC al 95% (promesa: 95%)  |  n = {n:,}")
print(f"{'alpha':>8} | {'Media verdadera':>16} | {'Cobertura real':>15} | {'Var finita?':>12}")
print("-" * 60)
for a in alphas:
    cov, mu = coverage_experiment(a, n=n)
    var_finite = "Sí" if a > 2 else "NO ⚠️"
    print(f"{a:>8.1f} | {mu:>16.4f} | {cov:>14.1%} | {var_finite:>12}")

print("\nObserva: cuando alpha ≤ 2 (varianza infinita), la cobertura cae por debajo del 95%")
print("El IC produce un número, pero ese número ya no tiene la interpretación prometida.")

In [ ]:
# 🔧 Experimento: juega con alpha y observa el comportamiento
# Prueba alpha = 5, 3, 2.5, 2.0, 1.8, 1.5, 1.2, 1.0 (Cauchy)
# Ve de mayor a menor para ver la transición
alpha = 1.5  # <-- CHANGE THIS

n_per_trial = 2000  # <-- CHANGE THIS
n_trials = 600

def draw_fat(alpha, n, seed):
    rng_local = np.random.default_rng(seed)
    if alpha <= 1.0:
        return rng_local.standard_cauchy(n)
    else:
        # Standard Pareto: P(X>x) ~ x^{-alpha}, x>=1
        # E[X] = alpha/(alpha-1), Var[X] finite iff alpha > 2
        return rng_local.pareto(alpha, size=n) + 1

estimates = np.array([draw_fat(alpha, n_per_trial, i).mean() for i in range(n_trials)])
sigmas    = np.array([draw_fat(alpha, n_per_trial, i).std()  for i in range(n_trials)])
true_mean = alpha / (alpha - 1) if alpha > 1 else np.nan

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Distribution of estimates (clipped for readability)
p1, p99 = np.percentile(estimates, [1, 99])
clipped = np.clip(estimates, p1 - 0.5*(p99-p1), p99 + 0.5*(p99-p1))
axes[0].hist(clipped, bins=60, color="#2E86AB", alpha=0.7, edgecolor="white", density=True)
if not np.isnan(true_mean):
    axes[0].axvline(true_mean, color="red", lw=2, ls="--",
                    label=f"Media verdadera = {true_mean:.2f}")
    axes[0].legend(fontsize=9)
axes[0].set_title(f"Distribución de $\\hat{{\\mu}}_n$ (vista recortada)\n$\\alpha={alpha}$, $n={n_per_trial:,}$")
axes[0].set_xlabel("Estimado")

# 2. Running average for 5 sequences
axes[1].set_title(f"Promedio acumulado (5 corridas)\n$\\alpha={alpha}$")
colors_5 = ["#2E86AB", "#E94F37", "#27AE60", "#F39C12", "#8E44AD"]
ns_seq = np.arange(1, n_per_trial + 1)
all_running = []
for j, col in enumerate(colors_5):
    seq = draw_fat(alpha, n_per_trial, j + 9000)
    running = np.cumsum(seq) / ns_seq
    all_running.append(running)
    axes[1].semilogx(ns_seq, running, lw=0.9, color=col, alpha=0.85)
if not np.isnan(true_mean):
    axes[1].axhline(true_mean, color="black", lw=1.5, ls="--",
                    label=f"Media = {true_mean:.2f}")
    axes[1].legend(fontsize=9)
    # Reasonable y-range (exclude extreme outliers from view)
    all_flat = np.concatenate(all_running)
    axes[1].set_ylim(max(0, np.percentile(all_flat, 2)),
                     np.percentile(all_flat, 98))
axes[1].set_xlabel("n (log)")
axes[1].set_ylabel("Promedio acumulado")

# 3. sigma_hat instability
axes[2].plot(np.sort(sigmas), ".", ms=2, color="#E94F37", alpha=0.6)
axes[2].set_title(f"$\\hat{{\\sigma}}$ muestral por experimento (ordenado)\n$\\alpha={alpha}$")
axes[2].set_xlabel("Experimento (ordenado)")
axes[2].set_ylabel("$\\hat{\\sigma}$")
if sigmas.max() / (sigmas.min() + 1e-10) > 10:
    axes[2].set_yscale("log")

plt.tight_layout()
plt.show()

# Diagnosis
print(f"alpha = {alpha}")
if not np.isnan(true_mean):
    print(f"Media verdadera:        {true_mean:.4f}")
    print(f"Mediana de estimados:   {np.median(estimates):.4f}")
    print(f"Promedio de estimados:  {estimates.mean():.4f}")
    print(f"Rango p5–p95:           [{np.percentile(estimates,5):.3f}, {np.percentile(estimates,95):.3f}]")
else:
    print("Media verdadera: NO EXISTE (Cauchy/alpha ≤ 1)")
    print(f"Mediana de estimados:   {np.median(estimates):.4f}  (sigue siendo ~0 para Cauchy)")

if alpha > 2:
    print("\n✓ Varianza finita (alpha > 2)  — CLT aplica, IC ~95% válido")
elif alpha > 1:
    print("\n⚠️  Varianza INFINITA (1 < alpha ≤ 2)")
    print("   LLN sí aplica (el promedio converge), pero muy lentamente.")
    print("   CLT no aplica → IC al 95% tiene cobertura real < 95%.")
else:
    print("\n✗ Media INFINITA (alpha ≤ 1, e.g. Cauchy)")
    print("   LLN falla: el promedio nunca converge.")
    print("   Hablar de 'la media MC' no tiene sentido.")

---
## Sección 4: Primer vistazo — Reducción de Varianza

El error del estimador MC es $\sigma/\sqrt{n}$. Para reducir el error a la mitad:
- **Opción A**: 4× más muestras (costoso)
- **Opción B**: reducir $\sigma$ (ingenioso)

La **reducción de varianza** explota la segunda opción: con el mismo presupuesto de muestras, obtenemos estimados más precisos.

### Antithetic Variates: la idea más simple

Si $U \sim \text{Uniform}(0,1)$, entonces $1-U$ también es $\text{Uniform}(0,1)$.

En vez de usar solo $f(U)$, podemos usar el promedio $\frac{f(U) + f(1-U)}{2}$.

Si $f$ es monótona, entonces $\text{Cov}(f(U), f(1-U)) \leq 0$, y:

$$\text{Var}\left[\frac{f(U)+f(1-U)}{2}\right] = \frac{\text{Var}[f(U)] + \text{Cov}(f(U),f(1-U))}{2} \leq \frac{\text{Var}[f(U)]}{2}$$

**Resultado**: con el mismo número de evaluaciones de $f$, la varianza se reduce.

In [ ]:
# Estimate integral of sin(pi*x) over [0,1]
# True value: integral of sin(pi*x) dx from 0 to 1 = 2/pi ≈ 0.6366
true_val_sin = 2 / np.pi

def crude_mc(f, n, seed=None):
    rng_local = np.random.default_rng(seed)
    u = rng_local.uniform(0, 1, n)
    return f(u).mean()

def antithetic_mc(f, n, seed=None):
    """Antithetic variates: use U and 1-U together."""
    rng_local = np.random.default_rng(seed)
    # n/2 pairs (u, 1-u)
    u = rng_local.uniform(0, 1, n // 2)
    pairs = (f(u) + f(1 - u)) / 2   # average of antithetic pair
    return pairs.mean()

f_sin = lambda u: np.sin(np.pi * u)
n = 1_000
n_experiments = 2_000

# Run many experiments
crude_ests = [crude_mc(f_sin, n, seed=i) for i in range(n_experiments)]
anti_ests = [antithetic_mc(f_sin, n, seed=i) for i in range(n_experiments)]

crude_var = np.var(crude_ests)
anti_var = np.var(anti_ests)

print(f"Integral verdadera: {true_val_sin:.6f}")
print(f"MC Crudo   — varianza: {crude_var:.6f}")
print(f"Antithetic — varianza: {anti_var:.6f}")
print(f"Reducción de varianza: {(1 - anti_var/crude_var)*100:.1f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

bins = np.linspace(min(min(crude_ests), min(anti_ests)),
                   max(max(crude_ests), max(anti_ests)), 60)
ax1.hist(crude_ests, bins=bins, density=True, alpha=0.6, color="#E94F37", label="MC Crudo")
ax1.hist(anti_ests, bins=bins, density=True, alpha=0.6, color="#2E86AB", label="Antithetic")
ax1.axvline(true_val_sin, color="black", ls="--", lw=1.5, label="Verdadero")
ax1.set_xlabel("Estimado")
ax1.set_title("Distribución de estimados")
ax1.legend()

ax2.boxplot([crude_ests, anti_ests], labels=["MC Crudo", "Antithetic"],
            patch_artist=True,
            boxprops=dict(facecolor="lightblue"),
            medianprops=dict(color="red", lw=2))
ax2.axhline(true_val_sin, color="black", ls="--", lw=1.5)
ax2.set_title(f"Boxplots — mismas {n:,} evaluaciones de f")
ax2.set_ylabel("Estimado")

plt.tight_layout()
plt.show()

### ¿Y si f no es monótona?

Antithetic variates solo reduce la varianza garantizadamente cuando $f$ es monótona. Para $f(x) = \sin(4\pi x)$, que oscila cuatro veces en $[0,1]$, la reducción puede no funcionar o incluso invertirse.

Esto nos lleva a la pregunta natural: **¿existen métodos más sofisticados de reducción de varianza?**

La respuesta es sí — y los exploraremos en profundidad en el **Notebook 02: Reducción de Varianza**, donde cubriremos:

- **Antithetic variates** (con el análisis completo de cuándo funciona y cuándo no)
- **Control variates** (usar una función correlacionada que sí puedes integrar analíticamente)
- **Importance sampling** (muestrear de una distribución distinta y re-ponderar)

Cada una tiene sus supuestos, sus ventajas, y sus casos de fallo — y los estudiaremos todos.

---

### Resumen de este notebook

| Concepto | Lo que vimos |
|---------|-------------|
| Estimador MC | $\hat{\mu}_n = \frac{1}{n}\sum f(X_i)$, muestrea y promedia |
| LLN | El promedio converge al valor verdadero (casi seguramente) |
| CLT | El error tiene distribución normal con $\text{SE} = \sigma/\sqrt{n}$ |
| IC 95% | $\hat{\mu}_n \pm 1.96 \cdot \hat{\sigma}/\sqrt{n}$ |
| Dimensionalidad | El costo de MC no depende de $d$ — cuadratura sí |
| Antithetic | Mismo $n$, menor $\sigma^2$, cuando $f$ es monótona |

**Siguiente paso**: [Notebook 02 — Reducción de Varianza](02_reduccion_varianza.ipynb) para ir más a fondo en cómo reducir $\sigma$.